# Day 8: Snowflake Storage Integration & Dynamic Tables

This notebook uses DuckDB to simulate Snowflake's external tables and incremental dynamic tables behaviour.

In [ ]:
import duckdb
import pandas as pd

con = duckdb.connect(database=':memory:')

# Create mock 'external' data (simulating Iceberg table in S3)
mock_external_data = pd.DataFrame({
    'person_id': ['pat_1', 'pat_2'],
    'condition_start_date': ['2023-01-01', '2023-02-15'],
    'condition_source_value': ['J45', 'E11']
})

con.execute("CREATE TABLE iceberg_ext AS SELECT * FROM mock_external_data")


In [ ]:
# Simulate Dynamic Table Creation (Materialized View in DuckDB)
con.execute("""
CREATE VIEW dt_condition_occurrence AS
SELECT * FROM iceberg_ext;
""")

print("Initial Dynamic Table state:")
print(con.execute("SELECT * FROM dt_condition_occurrence").df())


In [ ]:
# Simulate incremental data arrival in 'Iceberg'
new_data = pd.DataFrame({
    'person_id': ['pat_3'],
    'condition_start_date': ['2023-03-01'],
    'condition_source_value': ['I10']
})
con.execute("INSERT INTO iceberg_ext SELECT * FROM new_data")

# View automatically 'updates' when queried, simulating dynamic table refresh
print("\nDynamic Table state after external data update:")
print(con.execute("SELECT * FROM dt_condition_occurrence").df())
